# 05.1 Personalized Recommendation System — BGE Embeddings

Modern-method counterpart to `05_Personalized_Recommendation_System_ABSA.ipynb`.

**Goal:** evaluate modern instruction-tuned embedding models for aspect-based coffee retrieval and compare against the TF-IDF and SBERT (MiniLM) baselines from notebook 05.

**Models compared here:**
- `BAAI/bge-base-en-v1.5` — top-tier MTEB retrieval embedding (Aug 2023, 768d)
- `intfloat/e5-base-v2` — alternative modern retrieval embedding (768d)

**Shared with 05:** dataset, split, metrics (Precision@K, Recall@K, NDCG@K, MAP@K), `RANDOM_STATE=42`, `TOP_K=10`.

**Requirements:** `pip install -U sentence-transformers` (>= 2.7).

In [1]:
import re
import json
import os
import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from sentence_transformers import SentenceTransformer

In [2]:
DATA_PATH = 'Data/final_coffee_reviews_absa.csv'
RANDOM_STATE = 42
TOP_K = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACT_DIR = 'artifacts/recommendation_bge'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print('Device:', DEVICE)

Device: cuda


In [3]:
df = pd.read_csv(DATA_PATH)

if 'combined_text' not in df.columns:
    raise ValueError('ABSA combined_text column missing. Re-run notebook 01 preprocessing.')

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['combined_text'] = df['combined_text'].fillna('').apply(clean_text)
df['origin_country'] = df['Country'].astype(str).str.strip()

df = df[(df['combined_text'].str.len() > 5) & (df['origin_country'] != '')]

counts = df['origin_country'].value_counts()
valid = counts[counts >= 2].index
df = df[df['origin_country'].isin(valid)].reset_index(drop=True)

print(f'Rows: {len(df)}, Classes: {df["origin_country"].nunique()}')

Rows: 7577, Classes: 41


In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['origin_country']
)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

Train: 6061, Test: 1516


In [5]:
def precision_at_k(relevance, k):
    return np.sum(relevance[:k]) / k

def recall_at_k(relevance, total_relevant, k):
    if total_relevant == 0:
        return 0
    return np.sum(relevance[:k]) / total_relevant

def dcg_at_k(relevance, k):
    relevance = np.array(relevance[:k])
    return np.sum(relevance / np.log2(np.arange(2, len(relevance)+2)))

def ndcg_at_k(relevance, total_relevant, k):
    ideal = dcg_at_k([1]*min(total_relevant, k), k)
    if ideal == 0:
        return 0
    return dcg_at_k(relevance, k) / ideal

def average_precision_at_k(relevance, total_relevant, k):
    score = 0
    hits = 0
    for i in range(k):
        if relevance[i]:
            hits += 1
            score += hits / (i+1)
    return score / min(total_relevant, k) if total_relevant > 0 else 0

def evaluate(query_vecs, query_labels, corpus_vecs, corpus_labels, k=TOP_K):
    sim = cosine_similarity(query_vecs, corpus_vecs)
    results = []
    for i in range(len(query_labels)):
        sims = sim[i]
        idx = np.argsort(-sims)
        ranked_labels = corpus_labels.iloc[idx].values
        true_label = query_labels.iloc[i]
        relevance = [1 if l == true_label else 0 for l in ranked_labels[:k]]
        total_relevant = int(np.sum(corpus_labels == true_label))
        results.append({
            'precision': precision_at_k(relevance, k),
            'recall': recall_at_k(relevance, total_relevant, k),
            'ndcg': ndcg_at_k(relevance, total_relevant, k),
            'map': average_precision_at_k(relevance, total_relevant, k)
        })
    return pd.DataFrame(results).mean()

## Baselines (reproduced from 05)

TF-IDF and SBERT (MiniLM) recomputed here so all results sit in one table.

In [6]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
train_vec = tfidf.fit_transform(train_df['combined_text'])
test_vec = tfidf.transform(test_df['combined_text'])

metrics_tfidf = evaluate(
    test_vec, test_df['origin_country'],
    train_vec, train_df['origin_country']
)
print('=== TF-IDF ===')
print(metrics_tfidf)

=== TF-IDF ===
precision    0.168668
recall       0.002858
ndcg         0.175958
map          0.090984
dtype: float64


In [7]:
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
train_emb_sbert = sbert.encode(train_df['combined_text'].tolist(), convert_to_numpy=True, show_progress_bar=True, batch_size=64)
test_emb_sbert  = sbert.encode(test_df['combined_text'].tolist(),  convert_to_numpy=True, show_progress_bar=True, batch_size=64)

metrics_sbert = evaluate(
    test_emb_sbert, test_df['origin_country'],
    train_emb_sbert, train_df['origin_country']
)
print('=== SBERT (MiniLM-L6) ===')
print(metrics_sbert)

del sbert; torch.cuda.empty_cache() if DEVICE == 'cuda' else None

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/95 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

=== SBERT (MiniLM-L6) ===
precision    0.165172
recall       0.002705
ndcg         0.170470
map          0.088605
dtype: float64


## BGE-base-en-v1.5

Uses the recommended retrieval prompt: queries get prefixed with `"Represent this sentence for searching relevant passages: "`; corpus passages are encoded plain. This asymmetric setup is what BGE was trained for.

In [8]:
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

bge = SentenceTransformer('BAAI/bge-base-en-v1.5', device=DEVICE)

train_emb_bge = bge.encode(
    train_df['combined_text'].tolist(),
    convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True, batch_size=32
)
test_emb_bge = bge.encode(
    [BGE_QUERY_PREFIX + t for t in test_df['combined_text'].tolist()],
    convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True, batch_size=32
)

metrics_bge = evaluate(
    test_emb_bge, test_df['origin_country'],
    train_emb_bge, train_df['origin_country']
)
print('=== BGE-base-en-v1.5 ===')
print(metrics_bge)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/190 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

=== BGE-base-en-v1.5 ===
precision    0.179420
recall       0.002972
ndcg         0.185617
map          0.101789
dtype: float64


## E5-base-v2 (secondary modern model)

E5 uses `query: ` / `passage: ` prefixes as part of its training — not optional.

In [9]:
try:
    del bge; torch.cuda.empty_cache() if DEVICE == 'cuda' else None

    e5 = SentenceTransformer('intfloat/e5-base-v2', device=DEVICE)

    train_emb_e5 = e5.encode(
        ['passage: ' + t for t in train_df['combined_text'].tolist()],
        convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=True, batch_size=32
    )
    test_emb_e5 = e5.encode(
        ['query: ' + t for t in test_df['combined_text'].tolist()],
        convert_to_numpy=True, normalize_embeddings=True,
        show_progress_bar=True, batch_size=32
    )

    metrics_e5 = evaluate(
        test_emb_e5, test_df['origin_country'],
        train_emb_e5, train_df['origin_country']
    )
    print('=== E5-base-v2 ===')
    print(metrics_e5)

    del e5; torch.cuda.empty_cache() if DEVICE == 'cuda' else None

except Exception as e:
    print('E5 skipped:', e)
    metrics_e5 = None

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/190 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

=== E5-base-v2 ===
precision    0.180409
recall       0.003027
ndcg         0.186564
map          0.101143
dtype: float64


## Summary table

In [10]:
summary_rows = {
    'TF-IDF (baseline)': metrics_tfidf,
    'SBERT MiniLM-L6 (baseline)': metrics_sbert,
    'BGE-base-en-v1.5': metrics_bge,
}
if metrics_e5 is not None:
    summary_rows['E5-base-v2'] = metrics_e5

summary = pd.DataFrame(summary_rows).T
summary = summary[['precision','recall','ndcg','map']]
print(summary.round(4))

summary.to_csv(os.path.join(ARTIFACT_DIR, 'recommendation_summary.csv'))
with open(os.path.join(ARTIFACT_DIR, 'recommendation_summary.json'), 'w') as f:
    json.dump({k: v.to_dict() for k, v in summary_rows.items()}, f, indent=2)
print('\nSaved to', ARTIFACT_DIR)

                            precision  recall    ndcg     map
TF-IDF (baseline)              0.1687  0.0029  0.1760  0.0910
SBERT MiniLM-L6 (baseline)     0.1652  0.0027  0.1705  0.0886
BGE-base-en-v1.5               0.1794  0.0030  0.1856  0.1018
E5-base-v2                     0.1804  0.0030  0.1866  0.1011

Saved to artifacts/recommendation_bge


## Qualitative: BGE recommendation demo

In [11]:
bge = SentenceTransformer('BAAI/bge-base-en-v1.5', device=DEVICE)

def recommend_coffee_bge(query, top_n=5):
    q = clean_text(query)
    q_emb = bge.encode(
        [BGE_QUERY_PREFIX + q],
        convert_to_numpy=True, normalize_embeddings=True
    )
    sims = cosine_similarity(q_emb, train_emb_bge)[0]
    idx = np.argsort(-sims)[:top_n]
    result = train_df.iloc[idx][[
        'Coffee Name', 'Roaster', 'origin_country', 'Blind Assessment'
    ]].copy()
    result.insert(0, 'similarity', sims[idx])
    return result.reset_index(drop=True)

query = 'chocolate caramel sweet smooth body'
print('=== BGE RECOMMENDATION ===')
print(recommend_coffee_bge(query, 5))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== BGE RECOMMENDATION ===
   similarity                                        Coffee Name  \
0    0.704676                         Big Truck Organic Brew Bag   
1    0.697020  100% Arabica Freeze-Dried Colombian (Instant C...   
2    0.692021                          Pomeranian Espresso Blend   
3    0.686980                             Keynote Perennial Brew   
4    0.683029                      Kochere Yirga Cheffe Espresso   

                 Roaster origin_country  \
0         Olympia Coffee       Ethiopia   
1            Waka Coffee       Colombia   
2             Buon Caffe       Ethiopia   
3       Chromatic Coffee         Brazil   
4  Coava Coffee Roasters       Ethiopia   

                                    Blind Assessment  
0  evaluated in brew bag format with a steeping t...  
1  evaluated at proportions of 5 grams of instant...  
2  evaluated as espresso. crisply sweet, chocolat...  
3  sweetly pungent, gentle, balanced. caramel, ce...  
4  evaluated as espresso. rich